<a href="https://colab.research.google.com/github/nihemelandu/causal-forecasting-pricing/blob/main/generating_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

SEED = 42

In [2]:
rng_data = np.random.default_rng(SEED)        # separate from the single-article rng

N_ARTICLES, N_WEEKS = 4467, 100
N_D, N_K, N_SEASON_GROUPS = 45, 15, 6

alpha_d = rng_data.normal(10, 3, N_D)         # Eq. 17: N(10, 3^2), 45 categories
beta_k  = rng_data.normal(300, 50, N_K)       # Eq. 19: N(300, 50^2), 15 categories

# Season shifts: k split into six subgroups, one integer shift in [-15, 15] each
season_groups = np.array_split(np.arange(N_K), N_SEASON_GROUPS)   # sizes 3,3,3,2,2,2
group_shift   = rng_data.integers(-15, 16, N_SEASON_GROUPS)       # upper bound is exclusive
k_shift = np.zeros(N_K, dtype=int)
for g, ks in enumerate(season_groups):
    k_shift[ks] = group_shift[g]

print("alpha_d: mean %.2f  sd %.2f  min %.2f  max %.2f" % (alpha_d.mean(), alpha_d.std(), alpha_d.min(), alpha_d.max()))
print("beta_k : mean %.1f  sd %.1f  min %.1f  max %.1f" % (beta_k.mean(), beta_k.std(), beta_k.min(), beta_k.max()))
for g, ks in enumerate(season_groups):
    print(f"season group {g}: k = {list(ks)}  shift = {group_shift[g]}")

alpha_d: mean 10.17  sd 2.36  min 4.15  max 16.42
beta_k : mean 304.7  sd 37.7  min 227.1  max 374.7
season group 0: k = [np.int64(0), np.int64(1), np.int64(2)]  shift = 11
season group 1: k = [np.int64(3), np.int64(4), np.int64(5)]  shift = 2
season group 2: k = [np.int64(6), np.int64(7), np.int64(8)]  shift = -13
season group 3: k = [np.int64(9), np.int64(10)]  shift = 8
season group 4: k = [np.int64(11), np.int64(12)]  shift = 2
season group 5: k = [np.int64(13), np.int64(14)]  shift = 4


In [3]:
d_names = [
    "T-Shirts", "Polo Shirts", "Shirts", "Blouses", "Sweaters", "Hoodies", "Cardigans",
    "Jackets", "Coats", "Blazers", "Jeans", "Trousers", "Chinos", "Shorts", "Skirts",
    "Dresses", "Jumpsuits", "Leggings", "Swimwear", "Underwear", "Socks", "Pajamas",
    "Sports Tops", "Sports Bottoms", "Sneakers", "Boots", "Sandals", "Heels", "Flats",
    "Loafers", "Handbags", "Backpacks", "Wallets", "Belts", "Scarves", "Hats", "Gloves",
    "Sunglasses", "Watches", "Jewelry", "Ties", "Perfume", "Kids Tops", "Kids Bottoms",
    "Kids Shoes",
]
assert len(d_names) == N_D == 45

d_lookup = {i: name for i, name in enumerate(d_names)}      # d (0..44) -> label
for i in range(5):
    print(i, d_lookup[i], f"alpha_d = {alpha_d[i]:.2f}")

0 T-Shirts alpha_d = 10.91
1 Polo Shirts alpha_d = 6.88
2 Shirts alpha_d = 12.25
3 Blouses alpha_d = 12.82
4 Sweaters alpha_d = 4.15


In [4]:
tiers = ["Core", "Fashion", "Basics"]
k_names = []
for g, ks in enumerate(season_groups):            # g = season group, ks = the k values in it
    for n, k in enumerate(ks):
        k_names.append(f"Season {chr(65 + g)} - {tiers[n]}")
assert len(k_names) == N_K == 15

k_lookup = {i: name for i, name in enumerate(k_names)}   # k (0..14) -> label
print(" k  name                  beta_k   season_shift")
for k in range(N_K):
    print(f"{k:>2}  {k_lookup[k]:<20} {beta_k[k]:>7.1f}   {k_shift[k]:>3}")

 k  name                  beta_k   season_shift
 0  Season A - Core        310.9    11
 1  Season A - Fashion     343.6    11
 2  Season A - Basics      311.2    11
 3  Season B - Core        333.9     2
 4  Season B - Fashion     303.4     2
 5  Season B - Basics      314.5     2
 6  Season C - Core        331.6   -13
 7  Season C - Fashion     227.1   -13
 8  Season C - Basics      284.0   -13
 9  Season D - Core        276.5     8
10  Season D - Fashion     268.1     8
11  Season E - Core        286.2     2
12  Season E - Fashion     374.7     2
13  Season F - Core        256.7     4
14  Season F - Fashion     348.4     4


In [5]:
def make_article(rng, max_draws=1000):
    d = int(rng.integers(0, N_D))                 # category d (random mapping)
    k = int(rng.integers(0, N_K))                 # category k (random mapping)
    t = np.arange(N_WEEKS)

    a_it = alpha_d[d] + rng.normal(0, 1, N_WEEKS)             # Eq. 16
    b_it = beta_k[k]  + rng.normal(0, 5, N_WEEKS)             # Eq. 18
    c_it = 0.05 * a_it**2 + 0.25 * a_it + 0.5 * b_it          # Eq. 15

    gamma_i     = rng.uniform(-0.02, 0.02)                    # Eq. 23
    sigma_tau_i = rng.uniform(0, 0.15)                        # Eq. 24
    tau_it = rng.normal(t * gamma_i, sigma_tau_i)             # Eq. 22
    s_it   = np.sin(2 * np.pi * (t + k_shift[k]) / 30)        # 30-week sine, shift tied to k

    q_b = (0.15 * tau_it + 0.25 * s_it + 1) * c_it            # Eq. 14

    e_b = max(1.3, rng.lognormal(0.75, 0.125))                # Eq. 20
    e_i = e_b * 0.15 * a_it.mean()

    q_b_bar = q_b.mean()
    for n_draws in range(1, max_draws + 1):                   # Eq. 21 + our valid-price rule
        p_i0 = rng.normal(q_b_bar / 3, q_b_bar / 1.5)
        if p_i0 > 0 and (q_b - p_i0 * e_i).min() >= 0:
            return dict(d=d, k=k, a=a_it, b=b_it, q_b=q_b, e_i=e_i, p_i0=p_i0,
                        n_draws=n_draws, gamma=gamma_i, sigma_tau=sigma_tau_i)
    return None                                               # dropped article

# Test on a separate generator so the main stream (rng_data) is untouched
rng_test = np.random.default_rng(123)
arts = [make_article(rng_test) for _ in range(1000)]
ok = [a for a in arts if a is not None]
draws = np.array([a['n_draws'] for a in ok])

print(f"articles made: {len(ok)} of 1000   dropped: {1000 - len(ok)}")
print(f"draws of p_i0 per article: mean {draws.mean():.1f}  median {np.median(draws):.0f}  max {draws.max()}")
print(f"mean q_b_bar {np.mean([a['q_b'].mean() for a in ok]):.1f}   mean e_i {np.mean([a['e_i'] for a in ok]):.2f}   mean p_i0 {np.mean([a['p_i0'] for a in ok]):.1f}")
print("\nfirst 5 articles:")
for a in ok[:5]:
    print(f"d={a['d']:>2} ({d_lookup[a['d']]:<14}) k={a['k']:>2} shift={k_shift[a['k']]:>3}  "
          f"q_b mean {a['q_b'].mean():6.1f}  e_i {a['e_i']:5.2f}  p_i0 {a['p_i0']:6.2f}  draws {a['n_draws']}")

articles made: 1000 of 1000   dropped: 0
draws of p_i0 per article: mean 8.7  median 6  max 94
mean q_b_bar 160.7   mean e_i 3.29   mean p_i0 18.0

first 5 articles:
d= 0 (T-Shirts      ) k=10 shift=  8  q_b mean  130.8  e_i  3.69  p_i0   9.58  draws 12
d= 0 (T-Shirts      ) k= 7 shift=-13  q_b mean  110.3  e_i  3.61  p_i0   6.42  draws 11
d=30 (Handbags      ) k= 8 shift=-13  q_b mean  157.5  e_i  5.54  p_i0  20.34  draws 21
d=28 (Flats         ) k=12 shift=  2  q_b mean  222.5  e_i  3.54  p_i0  39.79  draws 4
d=37 (Sunglasses    ) k=14 shift=  4  q_b mean  188.5  e_i  2.43  p_i0  20.79  draws 4


In [6]:
rng_lam = np.random.default_rng(321)
rows = []
for a in ok:
    r = run_policy(a, rng_lam.uniform(0, 1, N_WEEKS))
    assert r['stock'].min() >= -1e-9                                  # stock never negative
    assert r['demand'].min() >= 0                                     # demand never negative
    assert r['price'].max() <= a['p_i0'] + 1e-12                      # price never above p_i0
    assert r['j'].min() >= 0 and r['j'].max() <= 5                    # j in 0..5
    assert abs(r['z0'] - r['sales'].sum() - r['stock'][N_WEEKS]) < 1e-6   # stock books balance
    rows.append(summarize(a, r))
print("checks passed for all", len(rows), "articles\n")

S  = {k: np.array([r[k] for r in rows], dtype=float) for k in rows[0]}
z0s = np.array([(a['q_b'].mean() - 0.86 * a['p_i0'] * a['e_i']) * 100 for a in ok])
out = ~np.isnan(S['stockout_week'])
p = np.percentile

print(f"stocked out before end of week 99: {out.mean():.0%}")
if out.any():
    print(f"  stockout week (p10/p50/p90): {p(S['stockout_week'][out],10):.0f} / {p(S['stockout_week'][out],50):.0f} / {p(S['stockout_week'][out],90):.0f}")
if (~out).any():
    left = S['leftover'][~out] / z0s[~out]
    print(f"ended with stock left: {(~out).mean():.0%}; leftover as share of z0 (p50/p90): {p(left,50):.1%} / {p(left,90):.1%}")
share_unmet = S['unmet'] / S['total_demand']
print(f"unmet demand, share of total demand (p50/p90): {p(share_unmet,50):.1%} / {p(share_unmet,90):.1%}")
print(f"avg discount weeks 4-99 (p10/p50/p90): {p(S['disc_4_99'],10):.1%} / {p(S['disc_4_99'],50):.1%} / {p(S['disc_4_99'],90):.1%}")

NameError: name 'run_policy' is not defined

In [ ]:
import pandas as pd

if 'k_lookup' not in globals():                       # in case the k-name cell was not run
    tiers = ["Core", "Fashion", "Basics"]
    k_names = [f"Season {chr(65+g)} - {tiers[n]}" for g, ks in enumerate(season_groups) for n in range(len(ks))]
    k_lookup = {i: nm for i, nm in enumerate(k_names)}

frames, n_dropped = [], 0
while len(frames) < N_ARTICLES:
    art = make_article(rng_data)
    if art is None:
        n_dropped += 1
        continue
    r = run_policy(art, rng_data.uniform(0, 1, N_WEEKS))
    i = len(frames)
    frames.append(pd.DataFrame({
        "article_id": i,
        "week": np.arange(N_WEEKS),
        "d": art['d'], "d_name": d_lookup[art['d']],
        "k": art['k'], "k_name": k_lookup[art['k']],
        "promotion": int(rng_data.integers(0, 2)),         # our assumption: Bernoulli(0.5) per article, no effect on demand
        "p0": art['p_i0'],
        "price": r['price'],
        "discount": 1 - r['price'] / art['p_i0'],          # 0.0 .. 0.5
        "demand": r['demand'],                             # true demand
        "sales": r['sales'],                               # demand capped by stock
        "stock": r['stock'][:N_WEEKS],                     # stock at the start of the week
        "true_base_demand": art['q_b'],
        "true_effect_e": art['e_i'],
    }))

df = pd.concat(frames, ignore_index=True)
print("articles:", df['article_id'].nunique(), "  rows:", len(df), "  dropped during build:", n_dropped)
print(df.head(6).round(2).to_string())

In [ ]:
g = df.groupby('article_id')

# --- Structure ---
assert (g.size() == N_WEEKS).all()
assert not df.duplicated(['article_id', 'week']).any()
assert df[['price','discount','demand','sales','stock','true_base_demand']].notna().all().all()

# --- Ranges ---
assert df['discount'].min() >= -1e-9 and df['discount'].max() <= 0.5 + 1e-9
assert np.allclose(df['discount'] * 10, np.round(df['discount'] * 10), atol=1e-6)   # steps of 0.1
assert (df['price'] <= df['p0'] + 1e-9).all()
assert (df['demand'] >= 0).all() and (df['stock'] >= -1e-9).all()
assert (df['sales'] <= df['demand'] + 1e-9).all()

# --- Identities ---
assert np.allclose(df['demand'], df['true_base_demand'] - df['price'] * df['true_effect_e'])   # known true effect
nxt = g['stock'].shift(-1); ok_rows = nxt.notna()
assert np.allclose(nxt[ok_rows], (df['stock'] - df['sales'])[ok_rows])                          # stock books balance
print("structure, range and identity checks passed\n")

# --- Categories ---
first = df.drop_duplicates('article_id')
print(f"d used: {first['d'].nunique()} of {N_D}  (articles per d, min/max: {first['d'].value_counts().min()}/{first['d'].value_counts().max()})")
print(f"k used: {first['k'].nunique()} of {N_K}  (articles per k, min/max: {first['k'].value_counts().min()}/{first['k'].value_counts().max()})")

# --- Behaviour ---
df['stock_end'] = df['stock'] - df['sales']
zero_week = df[df['stock_end'] <= 1e-9].groupby('article_id')['week'].min()
print(f"\nstocked out by week 99: {len(zero_week)/N_ARTICLES:.0%}   stockout week p10/p50/p90: "
      f"{zero_week.quantile(.1):.0f} / {zero_week.quantile(.5):.0f} / {zero_week.quantile(.9):.0f}")
unmet = (df['demand'] - df['sales']).groupby(df['article_id']).sum() / g['demand'].sum()
print(f"unmet demand share p50/p90: {unmet.quantile(.5):.1%} / {unmet.quantile(.9):.1%}")
avg_disc = df[df['week'] >= 4].groupby('article_id')['discount'].mean()
print(f"avg discount weeks 4-99 p10/p50/p90: {avg_disc.quantile(.1):.1%} / {avg_disc.quantile(.5):.1%} / {avg_disc.quantile(.9):.1%}")
print("\nshare of article-weeks at each discount level:")
print(df['discount'].round(1).value_counts(normalize=True).sort_index().map("{:.1%}".format).to_string())

# --- Confounding: within an article, does the discount move against baseline demand? ---
late = df[df['week'] >= 4].copy()
late['dd'] = late['discount'] - late.groupby('article_id')['discount'].transform('mean')
late['bb'] = late['true_base_demand'] - late.groupby('article_id')['true_base_demand'].transform('mean')
num = (late['dd'] * late['bb']).groupby(late['article_id']).sum()
den = np.sqrt((late['dd']**2).groupby(late['article_id']).sum() * (late['bb']**2).groupby(late['article_id']).sum())
corr = (num / den).replace([np.inf, -np.inf], np.nan).dropna()
print(f"\ncorr(discount, base demand) within article, weeks 4-99: mean {corr.mean():.2f}, "
      f"share negative {(corr < 0).mean():.0%} (n={len(corr)})")

In [ ]:
df['season'] = np.sin(2 * np.pi * (df['week'] + k_shift[df['k'].to_numpy()]) / 30)   # same formula as the generator
mask = df['week'] >= 10                                   # same weeks for every lag so the results are comparable

def within_corr(a, b):
    d = df.loc[mask, ['article_id', a, b]]
    aa = d[a] - d.groupby('article_id')[a].transform('mean')
    bb = d[b] - d.groupby('article_id')[b].transform('mean')
    num = (aa * bb).groupby(d['article_id']).sum()
    den = np.sqrt((aa**2).groupby(d['article_id']).sum() * (bb**2).groupby(d['article_id']).sum())
    return (num / den).replace([np.inf, -np.inf], np.nan).dropna()

print("corr(discount this week, season L weeks earlier), within article, weeks 10-99")
for L in [0, 2, 4, 6, 8, 10]:
    df[f'season_lag{L}'] = df.groupby('article_id')['season'].shift(L)
    c = within_corr('discount', f'season_lag{L}')
    print(f"  L={L:>2}: mean {c.mean():+.2f}   share negative {(c < 0).mean():.0%}   (n={len(c)})")

In [ ]:
from pathlib import Path

cwd = Path.cwd()
data_dir = (cwd.parent if cwd.name == "notebooks" else cwd) / "data"
data_dir.mkdir(exist_ok=True)

cols = ["article_id", "week", "d", "d_name", "k", "k_name", "promotion", "p0", "price", "discount",
        "demand", "sales", "stock", "true_base_demand", "true_effect_e"]
out = df[cols]

try:
    path = data_dir / "synthetic_pricing_seed42.parquet"
    out.to_parquet(path, index=False)
except Exception as e:
    print("parquet not available:", e)
    path = data_dir / "synthetic_pricing_seed42.csv.gz"
    out.to_csv(path, index=False, compression="gzip")

print("saved:", path.resolve(), f"({path.stat().st_size / 1e6:.1f} MB)")
back = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
assert back.shape == out.shape and np.allclose(back['demand'], out['demand'])
print("read back OK:", back.shape)